# 🎬 Reels — Free-GPU Talking Avatar (Kaggle / MuseTalk)

Enable **Settings → Accelerator → GPU T4 x2** and **Internet: On**, then run top to bottom.

Output: a 9:16 `reel.mp4` you download from the Kaggle output panel.


In [ ]:
# 1) GPU check
!nvidia-smi -L
import torch;print('CUDA:',torch.cuda.is_available())

In [ ]:
# 2) Get repo
%cd /kaggle/working
!git clone https://github.com/amsinghnavdeep/reels.git 2>/dev/null || (cd reels && git pull)
%cd /kaggle/working/reels
!pip -q install edge-tts pydub pyyaml

In [ ]:
# 3) Clone + install MuseTalk (one-time)
import os
os.makedirs('engines',exist_ok=True)
if not os.path.isdir('engines/MuseTalk'):
    !git clone https://github.com/TMElyralab/MuseTalk.git engines/MuseTalk
%cd /kaggle/working/reels/engines/MuseTalk
!pip -q install -r requirements.txt
!pip -q install --no-cache-dir -U openmim
!mim install mmengine 'mmcv==2.0.1' 'mmdet==3.1.0' 'mmpose==1.1.0'
!mkdir -p ffmpeg && cd ffmpeg && wget -q https://johnvansickle.com/ffmpeg/releases/ffmpeg-release-amd64-static.tar.xz && tar xf ffmpeg-release-amd64-static.tar.xz --strip-components=1
os.environ['FFMPEG_PATH']='/kaggle/working/reels/engines/MuseTalk/ffmpeg'

In [ ]:
# 4) Download weights (~5 GB)
%cd /kaggle/working/reels/engines/MuseTalk
!sh ./download_weights.sh || bash ./download_weights.sh

In [ ]:
# 5) Script -> voice
%cd /kaggle/working/reels
SCRIPT='examples/script.txt'; AVATAR='avatars/panditji.png'; VOICE='hi-IN-MadhurNeural'
!python tts.py --script $SCRIPT --engine edge --voice $VOICE --output output/speech.wav
from IPython.display import Audio; Audio('output/speech.wav')

In [ ]:
# 6) MuseTalk lip-sync
import yaml,os
os.chdir('/kaggle/working/reels/engines/MuseTalk')
cfg={'task_0':{'video_path':'/kaggle/working/reels/'+AVATAR,'audio_path':'/kaggle/working/reels/output/speech.wav'}}
os.makedirs('configs/inference',exist_ok=True)
open('configs/inference/reels.yaml','w').write(yaml.dump(cfg))
!python -m scripts.inference --inference_config configs/inference/reels.yaml --result_dir /kaggle/working/reels/output/muse --version v15 --fps 25 --batch_size 4 --use_float16 --parsing_mode jaw --extra_margin 10

In [ ]:
# 7) 9:16 reel
%cd /kaggle/working/reels
import glob,os
clips=sorted(glob.glob('output/muse/**/*.mp4',recursive=True),key=os.path.getmtime)
assert clips,'No MuseTalk output — check previous cell logs.'
raw=clips[-1];print('raw:',raw)
!python reel_utils.py --in "$raw" --out output/reel.mp4
print('Download output/reel.mp4 from the Kaggle Output panel.')
from IPython.display import Video; Video('output/reel.mp4',embed=True,width=270)